# 05 — Live Module Demo (run top-to-bottom)

A single notebook that walks the **AMR Federated KG** codebase module by module.
Every section shows the real source of a module and then *runs it* on a sample of
the ARMD cohort.

**How to use**
1. Runtime → *Change runtime type* → **GPU** (T4/P100 is enough).
2. Run cells **top to bottom**. Slow cells take ~1–5 min each (graph build,
   GNN training, FedAvg simulation) — for a demo video, just let them run and
   fast-forward during playback.
3. `DEMO_ROWS` below controls the data size. 200k is a good live-demo balance;
   lower to 50–100k to make it snappier (numbers will differ from the logged
   full-scale runs — see the summary at the bottom).

Full-scale logged results (1.60M rows, multi-seed) live in `res.txt` and
`docs/2026-08-11-splits-and-results.md`.


## Setup — code + data

In [ ]:
# 1) Get the code (clone the branch, or pull if already cloned)
!git clone -b phase3-federated https://github.com/RawEgg6/Capstone-amr-fed.git 2>/dev/null || (cd Capstone-amr-fed && git fetch && git checkout phase3-federated && git pull)
!pip install -q torch_geometric 'flwr[simulation]'

In [ ]:
# 2) Point at the data. Mount Drive and set ARMD_DIR *before* importing amr_fed.
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['ARMD_DIR'] = '/content/drive/MyDrive/ARMD'   # EDIT to your ARMD folder

In [ ]:
import sys
from pathlib import Path

REPO = Path('/content/Capstone-amr-fed')
if not REPO.exists() and (Path.cwd() / 'src' / 'amr_fed').exists():
    REPO = Path.cwd()          # running from a local checkout
sys.path.insert(0, str(REPO / 'src'))
print('repo:', REPO)

In [ ]:
DEMO_ROWS = 200_000   # demo subset size (full-scale = 1.60M rows)
print('DEMO_ROWS =', DEMO_ROWS)

## Module 1 — `config.py` — the locked schema

*→ Open `src/amr_fed/config.py` in the file browser / editor and show it, then run the cell below.*

In [ ]:
# >>> run: config module
from amr_fed import config
D = Path(config.DATA_DIR)
print('DATA_DIR:', D, '| exists:', D.exists())
print('CSVs found:', sum((D / f).exists() for f in config.ARMD_TABLES.values()),
      'of', len(config.ARMD_TABLES))
print()
print('node types:', config.NODE_TYPES)
print('target edge (label):', config.TARGET_EDGE)
print('edge types:', len(config.EDGE_TYPES))
for e in config.EDGE_TYPES:
    print('   ', e)

## Module 2 — `data_loader.py` — one clean per-test frame

*→ Open `src/amr_fed/data_loader.py` (the `load_cohort_frame` function), then run the cell below.*

In [ ]:
# >>> run: data_loader module (loads + filters + labels + joins context tables)
from amr_fed.data_loader import load_cohort_frame, PK, ORG, ABX, CD
df = load_cohort_frame(sample_n=DEMO_ROWS)
print('rows:', f'{len(df):,}', '| organisms:', df[ORG].nunique(),
      '| antibiotics:', df[ABX].nunique())
print()
print('binary target  (0=Susceptible, 1=Resistant+Intermediate):')
print(df['label'].value_counts().sort_index().rename({0: 'S', 1: 'R+I'}).to_string())
print('positive rate (subset):', round(df['label'].mean(), 3))
print()
print('ward sizes (our partitioning axis):')
print(df['ward'].value_counts().to_string())
print()
print('sample rows:')
print(df[[PK, ORG, ABX, 'label', 'ward']].head(5).to_string(index=False))

#### Starting EDA — target balance & the ward / organism structure

In [ ]:
import matplotlib.pyplot as plt
lab = df['label'].map({0: 'Susceptible', 1: 'Resistant/Int.'}).value_counts()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(lab.index, lab.values, color=['#6baed6', '#d1495b'])
axes[0].set_title('Binary target balance (demo subset)')
for i, v in enumerate(lab.values):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')
wr = df.groupby('ward')['label'].agg(['mean', 'count']).sort_values('mean')
axes[1].bar(wr.index, wr['mean'], color='#3182bd')
axes[1].set_title('Resistance rate by ward')
axes[1].set_ylabel('P(resistant)')
for i, (r, c) in enumerate(zip(wr['mean'], wr['count'])):
    axes[1].text(i, r, f'{r:.2f}\n({c:,})', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Organisms matter most for resistance -> show the heavy hitters + their R-rate
top = df.groupby(ORG)['label'].agg(['mean', 'count']).sort_values('count', ascending=False).head(15)
order = top.iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 6))
cols = plt.cm.RdYlBu_r(order['mean'] / order['mean'].max())
ax.barh(order.index, order['count'], color=cols)
ax.set_xlabel('number of tests'); ax.set_title('Top organisms by tests (colour = resistance rate)')
for j, (org, row) in enumerate(order.iterrows()):
    ax.text(row['count'], j, f"  R-rate {row['mean']:.2f}", va='center', fontsize=8)
plt.tight_layout(); plt.show()

## Module 3 — `graph_build.py` — rows -> heterogeneous knowledge graph

*→ Open `src/amr_fed/graph_build.py` in the editor (`build_arrays`, `to_hetero_data`,
`build_graph` at the bottom), then run the cells below.*

In [ ]:
# >>> run: graph_build module  (pure arrays -> PyG HeteroData)
from amr_fed.graph_build import build_arrays, to_hetero_data
arrays = build_arrays(df, patient_history=True)   # leakage-safe patient-history features
data = to_hetero_data(arrays)
print(data)

In [ ]:
print('--- graph summary ---')
for nt in data.node_types:
    print(f'nodes [{nt:>10}] = {data[nt].num_nodes:>7,}   feature-dim = {data[nt].x.shape[1]}')
print()
for (s, r, d), ei in data.edge_index_dict.items():
    if not r.startswith('rev_'):
        print(f'edge  ({s:>10}, {r:>14}, {d:<9}) = {ei.shape[1]:>7,}')
print()
tr, va, te = data.train_mask.sum(), data.val_mask.sum(), data.test_mask.sum()
print(f'supervision triples: train {int(tr):,} | val {int(va):,} | test {int(te):,}')
print('patient-grouped split (no leakage): a patient is in exactly one split.')
if hasattr(data, 'triple_feat'):
    print('per-test decoder features (patient-history):', data.triple_feat.shape[1])

#### Visualising the KG structure we just built

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
nt = list(data.node_types)
axes[0].bar(range(len(nt)), [data[t].num_nodes for t in nt], color='#3182bd')
axes[0].set_xticks(range(len(nt))); axes[0].set_xticklabels(nt)
axes[0].set_title('Nodes by type'); axes[0].set_yscale('log')
for i, t in enumerate(nt):
    axes[0].text(i, data[t].num_nodes, f'{data[t].num_nodes:,}', ha='center', va='bottom', fontsize=8)
ed = {(s, r, d): ei.shape[1] for (s, r, d), ei in data.edge_index_dict.items() if not r.startswith('rev_')}
names = [f'{s}\n{r}\n{d}' for s, r, d in ed]; counts = list(ed.values())
axes[1].bar(range(len(names)), counts, color='#d1495b')
axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, fontsize=7)
axes[1].set_title('Edges by type')
for i, c in enumerate(counts):
    axes[1].text(i, c, f'{c:,}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# A tiny real neighbourhood from the loaded frame (patient -> organisms -> antibiotics)
import networkx as nx
cnt = df.groupby(PK).size()
pool = cnt[(cnt >= 3) & (cnt <= 8)]
p = pool.idxmax() if len(pool) else cnt.idxmax()
sub = df[df[PK] == p]
orgs = sub[ORG].unique().tolist(); abx = sub[ABX].unique().tolist()
G = nx.DiGraph()
G.add_node(p, kind='patient')
for o in orgs: G.add_node(o, kind='organism')
for a in abx:  G.add_node(a, kind='antibiotic')
for o in orgs:
    G.add_edge(p, o)
    for a in sub.loc[sub[ORG] == o, ABX]:
        G.add_edge(o, a)
pos = nx.spring_layout(G, seed=7)
cmap = {'patient': '#d1495b', 'organism': '#3182bd', 'antibiotic': '#31a354'}
nx.draw(G, pos, with_labels=True, node_color=[cmap[G.nodes[n]['kind']] for n in G],
        node_size=1800, font_size=7, arrows=True)
plt.title(f'One patient\'s neighbourhood in the KG (patient id {p})')
plt.tight_layout(); plt.show()

## Module 4 — `model.py` + `train_local.py` — AMR-SAGE training

*→ Open `src/amr_fed/model.py` (the `AMRSAGE` class) and `src/amr_fed/train_local.py`
(the `train` function), then run the cell below.*

Full scale (1.6M rows, 60 epochs) reaches **macro-F1 0.71 / AUROC 0.84** (majority
baseline 0.446). Here we train on the demo subset with a tiny budget just to show the
module running.*

In [ ]:
# >>> run: model + train_local modules (quick demo budget)
from amr_fed.train_local import train
model, metrics = train(data, hidden=64, layers=2, epochs=6, eval_every=3)
metrics

## Module 5 — `partition.py` — one dataset -> simulated hospitals

*→ Open `src/amr_fed/partition.py` (the 8 split functions), then run the cell below.*

In [ ]:
# >>> run: partition module — four different 'hospital' assignments, all on this frame
from amr_fed.partition import (dirichlet_ward_mixture, organism_community,
                               specimen_baseline, homophily_split)
seed = config.SEED
d = dirichlet_ward_mixture(df, n_clients=5, alpha=0.5, seed=seed)   # quantity/feature skew
o = organism_community(df, n_clients=5, seed=seed)                  # disjoint bug-sets
s = specimen_baseline(df)                                           # natural split by source
h = homophily_split(df, n_clients=4, seed=seed)                     # structure: clustering of R
assert d.equals(dirichlet_ward_mixture(df, n_clients=5, alpha=0.5, seed=seed))  # deterministic
for name, a in [('ward-Dirichlet alpha=0.5', d), ('organism-community', o),
                ('specimen', s), ('homophily spectrum', h)]:
    print(f'{name:>22}: hospital sizes {dict(a.value_counts().sort_index())}')
print()
print('no-leakage check (every patient assigned exactly once):',
      all(len(a) == len(df) and a.index.is_unique for a in (d, o, s, h)))

#### Hospital sizes under each split

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, a) in zip(axes, [('ward-Dirichlet', d), ('organism-community', o),
                                ('specimen', s), ('homophily', h)]):
    sz = a.value_counts().sort_index()
    ax.bar(sz.index.astype(str), sz.values, color='#6baed6')
    ax.set_title(name); ax.set_xlabel('hospital'); ax.set_ylabel('patients')
    ax.tick_params(axis='x', labelsize=7)
plt.tight_layout(); plt.show()

## Module 6 — the `federated/` package — Flower FedAvg baseline

*→ Open `src/amr_fed/federated/run.py` (the runner), `task.py` (shared client logic),
`client_app.py` (one simulated hospital), and `server_app.py` (FedAvg server), then run
the cells below.*

In [ ]:
import inspect
from amr_fed.federated.run import run_fedavg, headroom_gate
print('run_fedavg   ', inspect.signature(run_fedavg))
print('headroom_gate', inspect.signature(headroom_gate))

In [ ]:
# >>> run: the whole federated stack on the organism-community split (demo budget)
# (force-reload from disk so git pulls take effect — same pattern as notebook 04)
import sys as _s
for _m in [m for m in list(_s.modules) if m.startswith('amr_fed')]:
    del _s.modules[_m]
from amr_fed.federated.run import run_fedavg
from amr_fed.partition import organism_community

res = run_fedavg(partition_fn=organism_community, n_clients=5,
                 rounds=3, local_epochs=2, hidden=64, df=df,
                 label='organism-community (demo subset)')

#### The comparison, as a picture

In [ ]:
import numpy as np
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
loc = res['local_only_per_client']
pc  = [v if v is not None else float('nan') for v in res['fedavg_per_client']]
hosp = list(range(len(loc)))
w = 0.4
axes[0].bar([i - w/2 for i in hosp], loc, width=w, label='local-only', color='#9ecae1')
axes[0].bar([i + w/2 for i in hosp], pc,  width=w, label='FedAvg (best)', color='#d1495b')
axes[0].set_xticks(hosp); axes[0].set_ylabel('macro-F1')
axes[0].set_title(f"per-hospital macro-F1 (demo subset)\n"
                  f"pooled {res['pooled']:.3f} | local {res['local_only']:.3f} "
                  f"| FedAvg {res['fedavg_best']:.3f}")
axes[0].legend()
axes[1].plot(res['fed_by_round'], marker='o', color='#3182bd')
axes[1].set_title('FedAvg macro-F1 by round'); axes[1].set_xlabel('round')
axes[1].set_ylabel('macro-F1')
plt.tight_layout(); plt.show()

#### Optional — the headroom gate (runs a multi-seed FedAvg-vs-pooled comparison)

Roughly +5 min on the demo subset. Skip during a live demo — the full-scale result is
in the summary at the bottom and in `docs/2026-08-11-splits-and-results.md`.

In [ ]:
# OPTIONAL (slow-ish). topology_split = crossed homophily x degree -> 4 hospitals
import sys as _s2
for _m in [m for m in list(_s2.modules) if m.startswith('amr_fed')]:
    del _s2.modules[_m]
from functools import partial
from amr_fed.federated.run import headroom_gate
from amr_fed.partition import topology_split
gate = headroom_gate(partition_fn=partial(topology_split, purity=0.0),
                     n_clients=4, rounds=2, local_epochs=2,
                     seeds=(42,), df=df, label='topology-corners (demo subset)')
gate['pass']

## Module 7 — the offline unit tests (pure logic, no data/GPU)

In [ ]:
# Tests exercise the pure logic of data_loader / graph_build / partition
# (incl. the homophily + degree-skew splits the notebooks don't call directly).
import subprocess as _sp
print('--- offline unit tests ---')
for _t in ('test_data_loader', 'test_graph_build', 'test_partition'):
    print()
    print(f'$ python tests/{_t}.py')
    _r = _sp.run([sys.executable, str(REPO / 'tests' / f'{_t}.py')],
                 capture_output=True, text=True)
    print((_r.stdout or _r.stderr).strip())

## Full-scale logged results (context for the panel)

The cells above ran on a **demo subset** to prove each module executes. The
full-scale runs (1.60M rows, 3 seeds, `hidden=128`) are logged in `res.txt` and
`docs/2026-08-11-splits-and-results.md`:

| Setting | local-only | FedAvg-best | Pooled | Note |
|---|---|---|---|---|
| Ward-Dirichlet (α sweep) | ~0.69 | ~0.70 | ~0.71 | FedAvg beats local in 9/9 runs; recovers ~99% of pooled w/o sharing data |
| Specimen split | 0.696 | **0.715** | ~0.71 | clean win; worst hospital (urine) +0.023 |
| Organism-community | 0.696 | **0.719** | ~0.71 | strongest; worst hospital +0.037 |
| Label-Dirichlet | 0.678 | 0.601 | — | ❌ dead end (reported as a finding) |
| Homophily / degree-skew / corners / Louvain | 0.68–0.69 | 0.70 | 0.66–0.69 | **FedAvg beats pooled** — pooled size-weights hospitals, under-trains rare communities → this is the headroom the Phase-5 topology-aware aggregator targets |

Phase-1 local ceiling: **macro-F1 0.71 / AUROC 0.84** (patient-history features),
at/above the published Stanford baseline (AUROC 0.74–0.81).

Modules covered: `config` · `data_loader` · `graph_build` · `model` · `train_local`
· `partition` · `federated/run` · `federated/task` · `federated/client_app` ·
`federated/server_app` · `tests/`.
